<a href="https://colab.research.google.com/github/ArsalBAIG/Basic-Problems-of-PYTHON/blob/main/DBSCAN_Algo(Movie_Recomm).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing Libraries.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Loading DataSets.

In [ ]:
df_netflix = pd.read_csv('/content/titles.csv.zip')
df_hbo = pd.read_csv('/content/titles.csv (1).zip')
df_amazon = pd.read_csv('/content/titles.csv (2).zip')

In [ ]:
df = pd.concat([df_netflix, df_hbo, df_amazon], axis= 0)
df.head(3)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300


# Cleaning & Preprocessing

In [ ]:
df_movies = df.drop_duplicates()
df_movies.duplicated().sum()

np.int64(0)

In [ ]:
df_movies.head(3)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300


In [ ]:
df_movies.drop(['description', 'age_certification'], axis= 1, inplace= True)

/tmp/ipython-input-1753687091.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_movies.drop(['description', 'age_certification'], axis= 1, inplace= True)


In [ ]:
df['production_countries']

,production_countries
0,['US']
1,['US']
2,['US']
3,['GB']
4,"['GB', 'US']"
...,...
9866,['US']
9867,['US']
9868,['IN']
9869,[]


# Working with Production_countries

In [ ]:
# We're removing [, '', ]
df_movies.loc[:, 'production_countries'] = df_movies['production_countries'].str.replace(r"\[", '', regex= True).str.replace(r"'", '', regex= True).str.replace(r"\]", '', regex= True)
# Here, we're including just first Country instead of two.
df_movies.loc[:, 'lead_prod_country'] = df_movies['production_countries'].str.split(',').str[0]
# Getting the length.
df_movies.loc[:, 'production_countries_len'] = df_movies['production_countries'].str.split(',').str.len()
# For replacing 0 with Nan in lead_prod_country
df_movies.loc[:, 'lead_prod_country'] = df_movies['lead_prod_country'].replace('', np.nan)

In [ ]:
df_movies['lead_prod_country']

,lead_prod_country
0,US
1,US
2,US
3,GB
4,GB
...,...
9866,US
9867,US
9868,IN
9869,NaN


# Working with geners.

In [ ]:
df_movies['genres']

,genres
0,['documentation']
1,"['drama', 'crime']"
2,"['drama', 'action', 'thriller', 'european']"
3,"['fantasy', 'action', 'comedy']"
4,"['war', 'action']"
...,...
9866,['drama']
9867,['comedy']
9868,['crime']
9869,"['family', 'drama']"


In [ ]:
df_movies['genres'] = df_movies['genres'].str.replace(r"\[", '', regex= True).str.replace(r"'", '', regex= True).str.replace(r"\]", '', regex= True)
df_movies['lead_genre'] = df_movies['genres'].str.split(',').str[0]
df_movies['lead_genre'] = df_movies['lead_genre'].replace('', np.nan)

/tmp/ipython-input-2565344821.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_movies['genres'] = df_movies['genres'].str.replace(r"\[", '', regex= True).str.replace(r"'", '', regex= True).str.replace(r"\]", '', regex= True)
/tmp/ipython-input-2565344821.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_movies['lead_genre'] = df_movies['genres'].str.split(',').str[0]
/tmp/ipython-input-2565344821.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try

In [ ]:
df_movies['lead_genre']

,lead_genre
0,documentation
1,drama
2,drama
3,fantasy
4,war
...,...
9866,drama
9867,comedy
9868,crime
9869,family


In [ ]:
df_movies.drop(['genres', 'production_countries'], axis= 1, inplace= True)

/tmp/ipython-input-1732419400.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_movies.drop(['genres', 'production_countries'], axis= 1, inplace= True)


In [ ]:
df_movies.columns

Index(['id', 'title', 'type', 'release_year', 'runtime', 'seasons', 'imdb_id',
       'imdb_score', 'imdb_votes', 'tmdb_popularity', 'tmdb_score',
       'lead_prod_country', 'production_countries_len', 'lead_genre'],
      dtype='object')

# Dropping Duplicate values.

In [ ]:
df_movies.isnull().sum()

,0
id,0
title,1
type,0
release_year,0
runtime,0
seasons,14772
imdb_id,1394
imdb_score,1873
imdb_votes,1910
tmdb_popularity,670


In [ ]:
# Dropping missing values.
df_movies.dropna(inplace= True)
# Set 'title' as index.
df_movies.set_index('title', inplace=True)
# dropping 'id' & 'imdb' cols.
df_movies.drop(['id', 'imdb_id'], axis= 1, inplace= True)

/tmp/ipython-input-1646000409.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_movies.dropna(inplace= True)
/tmp/ipython-input-1646000409.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_movies.drop(['id', 'imdb_id'], axis= 1, inplace= True)


In [ ]:
df_movies.head(3)

,type,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,lead_prod_country,production_countries_len,lead_genre
title,,,,,,,,,,,
Monty Python's Flying Circus,SHOW,1969,30,4.0,8.8,73424.0,17.617,8.306,GB,1,comedy
Seinfeld,SHOW,1989,24,9.0,8.9,308824.0,130.213,8.301,US,1,comedy
Knight Rider,SHOW,1982,51,4.0,6.9,34115.0,50.267,7.500,US,1,scifi


# Encoding Categorical Cols/features.

In [ ]:
# converting categorical data into 0s & 1s.
dummys = pd.get_dummies(df_movies[['type', 'lead_genre', 'lead_prod_country']], drop_first= True)
# concatinating org data with dummy data.
df_movies_dummy = pd.concat([df_movies, dummys], axis= 1)
# dropping the dummy data columns.
df_movies_dummy.drop(['type', 'lead_genre', 'lead_prod_country'], axis= 1, inplace= True)

# Scalling

In [ ]:
df_movies_dummy.head(3)

,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,production_countries_len,lead_genre_animation,lead_genre_comedy,...,lead_prod_country_SG,lead_prod_country_SN,lead_prod_country_TH,lead_prod_country_TR,lead_prod_country_TW,lead_prod_country_UA,lead_prod_country_US,lead_prod_country_UY,lead_prod_country_XX,lead_prod_country_ZA
title,,,,,,,,,,,,,,,,,,,,,
Monty Python's Flying Circus,1969,30,4.0,8.8,73424.0,17.617,8.306,1,False,True,...,False,False,False,False,False,False,False,False,False,False
Seinfeld,1989,24,9.0,8.9,308824.0,130.213,8.301,1,False,True,...,False,False,False,False,False,False,True,False,False,False
Knight Rider,1982,51,4.0,6.9,34115.0,50.267,7.500,1,False,False,...,False,False,False,False,False,False,True,False,False,False


In [ ]:
# Making object of minmaxscalar.
scalar = MinMaxScaler()
# fit & transform data.
df_scalad = scalar.fit_transform(df_movies_dummy)
# Convert numpy array into dataframe.
df_scalad = pd.DataFrame(df_scalad, columns= df_movies_dummy.columns)
# Displaying
df_scalad.head(3)

,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,production_countries_len,lead_genre_animation,lead_genre_comedy,...,lead_prod_country_SG,lead_prod_country_SN,lead_prod_country_TH,lead_prod_country_TR,lead_prod_country_TW,lead_prod_country_UA,lead_prod_country_US,lead_prod_country_UY,lead_prod_country_XX,lead_prod_country_ZA
0,0.397727,0.168539,0.058824,0.9125,0.037009,0.007913,0.815870,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.625000,0.134831,0.156863,0.9250,0.155671,0.058490,0.815326,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.545455,0.286517,0.058824,0.6750,0.017194,0.022579,0.728261,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


# DBSCAN (Without Hyper-Parameters).

In [ ]:
db_values = [0.5, 0.10, 0.15]
min_sample_values = [5, 10, 15]

for epsilon in db_values:
  for min_sample in min_sample_values:
    clusterer = DBSCAN(eps= epsilon, min_samples= min_sample).fit(df_scalad)
    cluster_labels = clusterer.labels_ # This labels_ parameter is use to get labels from dbscan.

    if len(set(cluster_labels)) == 1: # In-case of only 1 label generated by dbscan.
      continue
    score = silhouette_score(df_scalad, cluster_labels)
    print('For epsilon= ', epsilon,
          'For min_samples= ', min_sample, # Add a comma here
          'Cluster count= ', len(set(cluster_labels)),
          'Silhoutte_Score= ', score)

For epsilon=  0.5 For min_samples=  5 Cluster count=  91 Silhoutte_Score=  0.601956050174035
For epsilon=  0.5 For min_samples=  10 Cluster count=  56 Silhoutte_Score=  0.5303679432698051
For epsilon=  0.5 For min_samples=  15 Cluster count=  40 Silhoutte_Score=  0.4842868784880638
For epsilon=  0.1 For min_samples=  5 Cluster count=  44 Silhoutte_Score=  0.06342426580267932
For epsilon=  0.1 For min_samples=  10 Cluster count=  23 Silhoutte_Score=  -0.07109094481959335
For epsilon=  0.1 For min_samples=  15 Cluster count=  15 Silhoutte_Score=  -0.10572992791946334
For epsilon=  0.15 For min_samples=  5 Cluster count=  61 Silhoutte_Score=  0.3056232791466584
For epsilon=  0.15 For min_samples=  10 Cluster count=  30 Silhoutte_Score=  0.2625389676014729
For epsilon=  0.15 For min_samples=  15 Cluster count=  21 Silhoutte_Score=  0.20269129553820267


# DBSCAN With best Hyper-Parameters.

In [ ]:
dbscan_model = DBSCAN(eps= 1, min_samples= 5).fit(df_scalad)
print('Cluster Count = ', len(set(dbscan_model.labels_)))
print('Silhoutte Score = ', silhouette_score(df_scalad, dbscan_model.labels_))


Cluster Count =  93
Silhoutte Score =  0.6091664186394288


In [ ]:
df_movies['dbscan_clusters'] = dbscan_model.labels_
df_movies.head(3)

/tmp/ipython-input-3795826599.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_movies['dbscan_clusters'] = dbscan_model.labels_


,type,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,lead_prod_country,production_countries_len,lead_genre,dbscan_clusters
title,,,,,,,,,,,,
Monty Python's Flying Circus,SHOW,1969,30,4.0,8.8,73424.0,17.617,8.306,GB,1,comedy,0
Seinfeld,SHOW,1989,24,9.0,8.9,308824.0,130.213,8.301,US,1,comedy,1
Knight Rider,SHOW,1982,51,4.0,6.9,34115.0,50.267,7.500,US,1,scifi,2


# Creating Function

In [ ]:
import random
# movie_name is our input.
def recommend_movies(movie_name: str):
    movie_name = movie_name.lower()
    df_movies['names'] = df_movies.index.str.lower() # Here, we place the movie's name that are stored at title place. Now store in-place of 'name' col.
    # Matching input with already exsisting names.
    actual_movie = df_movies[df_movies['names'].str.contains(movie_name, na= False)]

    if not actual_movie.empty:
      # Use iloc[0] to access the first element of the Series
      cluster = actual_movie['dbscan_clusters'].iloc[0]  # Her, we're extracting clusters that exsists in actual_movie.
      clustered_movie = df_movies[df_movies['dbscan_clusters']== cluster]

      if len(clustered_movie) >= 5:
        recomm_movie = random.sample(list(clustered_movie.index), 5)
      else:
        recomm_movie = list(clustered_movie.index)

      print('--- We can recommend you these movies ---') # Moved this line outside the else block
      for m in recomm_movie: # Changed recomm_movies to recomm_movie
          print(m)
    else:
          print('Movie not found in the database.')

In [ ]:
# Test1
input_movie = input('Enter any movie name ? ')
recommend_movies(input_movie)

Enter any movie name ? 
--- We can recommend you these movies ---
Man vs. Bee
Pramface
Chewing Gum
Man Like Mobeen
Almost Royal


/tmp/ipython-input-555349836.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_movies['names'] = df_movies.index.str.lower() # Here, we place the movie's name that are stored at title place. Now store in-place of 'name' col.


In [ ]:
# Test2.
input_movie = input('Enter any movie name ?')
recommend_movies(input_movie)

Enter any movie name ?hjhhj
Movie not found in the database.


/tmp/ipython-input-555349836.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_movies['names'] = df_movies.index.str.lower() # Here, we place the movie's name that are stored at title place. Now store in-place of 'name' col.


# Saving dataset.

In [ ]:
df_movies.to_csv('clustered_movies.csv', index= False)